In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS olist.silver")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Origem: olist.bronze.tb_customers
# Deduplicação via Window Function pelo timestamp_ingestion mais recente
df_customers = spark.table("olist.bronze.tb_customers")

# Window Function particionada por customer_id, ordenada pelo timestamp_ingestion
# mais recente para garantir que apenas o registro mais recente seja mantido
window_dedup = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

df_dim_consumidores = (
    df_customers
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank", "timestamp_ingestion")
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.col("customer_unique_id").alias("id_consumidor_unico"),
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado"),
    )
)

(
    df_dim_consumidores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.dim_consumidores")
)

print(f"✅ olist.silver.dim_consumidores ({df_dim_consumidores.count()} registros)")

In [0]:
# Origem: olist.bronze.tb_products
# Deduplicação via Window Function pelo timestamp_ingestion mais recente
df_products = spark.table("olist.bronze.tb_products")

window_dedup = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

df_dim_produtos = (
    df_products
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank", "timestamp_ingestion")
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_name_lenght").alias("tamanho_nome_produto"),
        F.col("product_description_lenght").alias("tamanho_descricao_produto"),
        F.col("product_photos_qty").alias("quantidade_fotos"),
        F.col("product_weight_g").alias("peso_produto_gramas"),
        F.col("product_length_cm").alias("comprimento_centimetros"),
        F.col("product_height_cm").alias("altura_centimetros"),
        F.col("product_width_cm").alias("largura_centimetros"),
        F.col("product_name").alias("nome_produto"),
    )
)

(
    df_dim_produtos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.dim_produtos")
)

print(f"✅ olist.silver.dim_produtos ({df_dim_produtos.count()} registros)")

In [0]:
# Origem: olist.bronze.tb_sellers
# Deduplicação via Window Function pelo timestamp_ingestion mais recente
df_sellers = spark.table("olist.bronze.tb_sellers")

window_dedup = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

df_dim_vendedores = (
    df_sellers
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank", "timestamp_ingestion")
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado"),
    )
)

(
    df_dim_vendedores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.dim_vendedores")
)

print(f"✅ olist.silver.dim_vendedores ({df_dim_vendedores.count()} registros)")

In [0]:
# Origem: olist.bronze.tb_orders
# Conversão do status de inglês para português
# Criação de colunas derivadas de tempo de entrega
df_orders = spark.table("olist.bronze.tb_orders")

mapa_status = {
    "delivered":   "entregue",
    "canceled":    "cancelado",
    "shipped":     "enviado",
    "processing":  "processando",
    "invoiced":    "faturado",
    "unavailable": "indisponível",
    "created":     "criado",
    "approved":    "aprovado",
}

status_expr = F.col("order_status")
for en, pt in mapa_status.items():
    status_expr = F.when(F.col("order_status") == en, pt).otherwise(status_expr)

df_fat_pedidos = (
    df_orders
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),
        status_expr.alias("status"),
        F.col("order_purchase_timestamp").alias("data_pedido"),
        F.col("order_approved_at").alias("data_aprovacao"),
        F.col("order_delivered_carrier_date").alias("data_entrega_transportadora"),
        F.col("order_delivered_customer_date").alias("data_entrega_real"),
        F.col("order_estimated_delivery_date").alias("data_entrega_estimada"),
    )
    .withColumn(
        "tempo_entrega_dias",
        F.datediff(F.col("data_entrega_real"), F.col("data_pedido"))
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(F.col("data_entrega_estimada"), F.col("data_pedido"))
    )
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )
    .withColumn(
        "entrega_no_prazo",
        F.when(F.col("status") != "entregue", "Não Entregue")
         .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
         .otherwise("Não")
    )
)

(
    df_fat_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.fat_pedidos")
)

print(f"✅ olist.silver.fat_pedidos ({df_fat_pedidos.count()} registros)")

In [0]:
# Origem: olist.bronze.tb_order_items

df_order_items = spark.table("olist.bronze.tb_order_items")

df_fat_itens_pedidos = (
    df_order_items
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("shipping_limit_date").alias("data_limite_envio"),
        F.col("price").alias("preco_BRL"),
        F.col("freight_value").alias("preco_frete"),
    )
)

(
    df_fat_itens_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.fat_itens_pedidos")
)

print(f"✅ olist.silver.fat_itens_pedidos ({df_fat_itens_pedidos.count()} registros)")

In [0]:
# Origem: olist.bronze.tb_order_payments
# Conversão do tipo de pagamento de inglês para português

df_order_payments = spark.table("olist.bronze.tb_order_payments")

mapa_pagamento = {
    "credit_card":  "Cartão de Crédito",
    "boleto":       "Boleto",
    "voucher":      "Voucher",
    "debit_card":   "Cartão de Débito",
    "not_defined":  "Não Definido",
}

pagamento_expr = F.col("payment_type")
for en, pt in mapa_pagamento.items():
    pagamento_expr = F.when(F.col("payment_type") == en, pt).otherwise(pagamento_expr)

df_fat_pagamentos_pedidos = (
    df_order_payments
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("sequencial_pagamento"),
        pagamento_expr.alias("tipo_pagamento"),
        F.col("payment_installments").alias("parcelas"),
        F.col("payment_value").alias("valor_pagamento"),
    )
)

(
    df_fat_pagamentos_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.fat_pagamentos_pedidos")
)

print(f"✅ olist.silver.fat_pagamentos_pedidos ({df_fat_pagamentos_pedidos.count()} registros)")

In [0]:
df_fat_avaliacoes = (
    df_reviews
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        # Cast explícito para integer na nota
        F.col("review_score").cast("integer").alias("nota_avaliacao"),
        F.col("review_comment_title").alias("titulo_avaliacao"),
        F.col("review_comment_message").alias("comentario_avaliacao"),
        F.try_to_timestamp(F.col("review_creation_date")).alias("data_criacao_avaliacao"),
        F.try_to_timestamp(F.col("review_answer_timestamp")).alias("data_resposta_avaliacao"),
    )
    .filter(F.col("id_pedido").isNotNull())
    .filter(
        F.col("data_criacao_avaliacao").isNull() |
        (F.col("data_criacao_avaliacao") <= F.current_timestamp())
    )
    .withColumn(
        "titulo_avaliacao",
        F.coalesce(F.col("titulo_avaliacao"), F.lit("Sem título"))
    )
    .withColumn(
        "comentario_avaliacao",
        F.coalesce(F.col("comentario_avaliacao"), F.lit("Sem comentário"))
    )
)

(
    df_fat_avaliacoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.fat_avaliacoes_pedidos")
)

print(f"✅ olist.silver.fat_avaliacoes_pedidos ({df_fat_avaliacoes.count()} registros)")

In [0]:
# Origem: olist.bronze.tb_product_category_name_translation
# Mapeamento de colunas para português e inglês

df_traducao = spark.table("olist.bronze.tb_product_category_name_translation")

df_dim_categoria_traducao = (
    df_traducao
    .select(
        F.col("product_category_name").alias("nome_produto_pt"),
        F.col("product_category_name_english").alias("nome_produto_en"),
    )
)

(
    df_dim_categoria_traducao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.dim_categoria_produtos_traducao")
)

print(f"✅ olist.silver.dim_categoria_produtos_traducao ({df_dim_categoria_traducao.count()} registros)")

In [0]:
# Origem: olist.bronze.tb_cotacao_dolar
# A API não fornece cotação para finais de semana, então criei um calendário
# contínuo e preenchemos as lacunas com a cotação de fechamento da sexta-feira
# anterior usando last(ignorenulls=True) sobre uma Window Function

df_cotacao = spark.table("olist.bronze.tb_cotacao_dolar")

# Converte a coluna de data e extrai apenas a data (sem hora)
df_cotacao = (
    df_cotacao
    .withColumn(
        "data_cotacao",
        F.to_date(F.col("dataHoraCotacao"))
    )
    # Mantém apenas a cotação mais recente do dia
    .groupBy("data_cotacao")
    .agg(F.last("cotacaoCompra").alias("cotacao_compra"))
)

# Cria calendário contínuo entre a data mínima e máxima da cotação
data_min, data_max = df_cotacao.agg(
    F.min("data_cotacao"),
    F.max("data_cotacao")
).first()

df_calendario = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{data_min}'),
        to_date('{data_max}'),
        interval 1 day
    )) AS data_cotacao
""")

# Faz o join do calendário com as cotações disponíveis
df_cotacao_completa = df_calendario.join(df_cotacao, "data_cotacao", "left")

# Window Function para preencher finais de semana com cotação da sexta anterior
# usando last(ignorenulls=True) conforme exigido pela atividade
window_fill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_dim_cotacao = (
    df_cotacao_completa
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(window_fill)
    )
)

(
    df_dim_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.dim_cotacao_dolar")
)

print(f"✅ olist.silver.dim_cotacao_dolar ({df_dim_cotacao.count()} registros)")

In [0]:
# Tabela final Silver: junção de fat_pedidos, fat_pagamentos_pedidos e dim_cotacao_dolar
# Valores financeiros arredondados para 2 casas decimais

df_fat_pedidos       = spark.table("olist.silver.fat_pedidos")
df_fat_pagamentos    = spark.table("olist.silver.fat_pagamentos_pedidos")
df_dim_cotacao       = spark.table("olist.silver.dim_cotacao_dolar")

df_pagamentos_agg = (
    df_fat_pagamentos
    .groupBy("id_pedido")
    .agg(F.sum("valor_pagamento").alias("valor_total_pago_brl"))
)

df_fat_pedido_total = (
    df_fat_pedidos
    .join(df_pagamentos_agg, "id_pedido", "left")
    .withColumn(
        "data_pedido_date",
        F.to_date(F.col("data_pedido"))
    )
    .join(
        df_dim_cotacao.select("data_cotacao", "cotacao_compra"),
        F.col("data_pedido_date") == F.col("data_cotacao"),
        "left"
    )
    .select(
        F.col("id_pedido"),
        F.col("id_consumidor"),
        F.col("status"),
        F.round("valor_total_pago_brl", 2).alias("valor_total_pago_brl"),
        F.round(
            F.col("valor_total_pago_brl") / F.col("cotacao_compra"), 2
        ).alias("valor_total_pago_usd"),
        F.col("data_pedido"),
    )
)

(
    df_fat_pedido_total.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.fat_pedido_total")
)

print(f"✅ olist.silver.fat_pedido_total ({df_fat_pedido_total.count()} registros)")

In [0]:
# Otimização com OPTIMIZE e ZORDER

tabelas_otimizar = [
    ("olist.silver.fat_pedido_total",      ["id_pedido", "data_pedido"]),
    ("olist.silver.fat_pedidos",           ["id_pedido", "data_pedido"]),
    ("olist.silver.fat_itens_pedidos",     ["id_pedido"]),
    ("olist.silver.fat_pagamentos_pedidos",["id_pedido"]),
    ("olist.silver.fat_avaliacoes_pedidos",["id_pedido"]),
]

for tabela, colunas in tabelas_otimizar:
    zorder_cols = ", ".join(colunas)
    spark.sql(f"OPTIMIZE {tabela} ZORDER BY ({zorder_cols})")
    print(f"✅ OPTIMIZE ZORDER aplicado em {tabela}")